In [ ]:
%%configure
{
  "defaultLakehouse": {
    "name": { "parameterName": "LakehouseName", "defaultValue": "AzureCostAnalyzer_LH" },
    "id": { "parameterName": "LakehouseId", "defaultValue": "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx" },
    "workspaceId": { "parameterName": "WorkspaceId", "defaultValue": "yyyyyyyy-yyyy-yyyy-yyyy-yyyyyyyyyyyy" }
  }
}

# 02 · Gold · Calendar

Builds **dim_date** and **dim_month** dynamically from the date range present in `focus_partitioned`.

| Output | Grain | Notes |
|---|---|---|
| `dim_date` | one row per day | covers `min(ChargePeriodStart) − 1 month` → `max(ChargePeriodEnd) + 1 month` |
| `dim_month` | one row per month | derived from `dim_date` |

**Mode**: overwrite each run. Safe because facts join by value (`Date` / `YearMonth`), not by surrogate key.

**Dependencies**: requires `focus_partitioned` (silver) — run after `01_Load_CostManagement_Focus_Data`.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

In [ ]:
# Parameters
SOURCE_TABLE      = "focus_partitioned"
DIM_DATE_TABLE    = "dim_date"
DIM_MONTH_TABLE   = "dim_month"
BUFFER_MONTHS_PRE = 1   # extend range backwards (for YoY/PY of earliest month)
BUFFER_MONTHS_POST= 1   # extend range forwards (for current-month-not-yet-billed cases)

## Step 1 · Discover date range from `focus_partitioned`

Pull min/max from `ChargePeriodStart` & `ChargePeriodEnd`. Add buffer months for time-intelligence.

In [ ]:
# Get raw min/max from focus_partitioned (with partition pruning)
# Assume focus_partitioned has partitions Year/Month; read last 18 months to be safe
from datetime import date
from dateutil.relativedelta import relativedelta

lookback_date = date.today() - relativedelta(months=18)

raw_dates = (
    spark.table(SOURCE_TABLE)
         .filter(
             (F.col("Year") > lookback_date.year) | 
             ((F.col("Year") == lookback_date.year) & (F.col("Month") >= lookback_date.month))
         )
         .select(
             F.min("ChargePeriodStart").alias("min_date"),
             F.max("ChargePeriodEnd").alias("max_date"),
         )
         .first()
)

raw_min = raw_dates.min_date
raw_max = raw_dates.max_date

# Snap to month boundaries and add buffer
spine_start = (raw_min.replace(day=1)) - relativedelta(months=BUFFER_MONTHS_PRE)
spine_end   = (raw_max.replace(day=1) + relativedelta(months=1) - timedelta(days=1)) + relativedelta(months=BUFFER_MONTHS_POST)

print(f"Raw range        : {raw_min} → {raw_max}")
print(f"Buffered range   : {spine_start} → {spine_end}")
print(f"Total days       : {(spine_end - spine_start).days + 1}")

## Step 2 · Build `dim_date`

Generate one row per day from spine_start to spine_end (inclusive).

In [ ]:
dim_date_df = spark.sql(f"""
    SELECT
        Date,
        YEAR(Date)                            AS Year,
        MONTH(Date)                           AS Month,
        DATE_FORMAT(Date, 'yyyy-MM')          AS YearMonth,
        QUARTER(Date)                         AS Quarter,
        CONCAT('Q', QUARTER(Date), ' ', YEAR(Date)) AS QuarterLabel,
        WEEKOFYEAR(Date)                      AS WeekOfYear,
        CONCAT(YEAR(Date), '-W', LPAD(WEEKOFYEAR(Date), 2, '0')) AS YearWeek,
        DAYOFMONTH(Date)                      AS DayOfMonth,
        DAYOFWEEK(Date)                       AS DayOfWeek,
        DATE_FORMAT(Date, 'EEEE')             AS DayName,
        DATE_FORMAT(Date, 'MMMM')             AS MonthName,
        CAST(DATE_FORMAT(Date, 'yyyyMMdd') AS INT) AS DateKey,
        CASE WHEN Date <= CURRENT_DATE() THEN TRUE ELSE FALSE END AS IsPast,
        CASE WHEN DATE_FORMAT(Date, 'yyyy-MM') = DATE_FORMAT(CURRENT_DATE(), 'yyyy-MM') THEN TRUE ELSE FALSE END AS IsCurrentMonth
    FROM (
        SELECT explode(sequence(DATE'{spine_start}', DATE'{spine_end}', INTERVAL 1 DAY)) AS Date
    )
""")

dim_date_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_DATE_TABLE)
print(f"✓ Created {DIM_DATE_TABLE} with {dim_date_df.count():,} rows")

## Step 3 · Build `dim_month`

Derive monthly grain from `dim_date` (one row per YearMonth).

In [ ]:
dim_month_df = (
    spark.table(DIM_DATE_TABLE)
         .groupBy("YearMonth")
         .agg(
             F.min("Date").alias("MonthStart"),
             F.first("Year").alias("Year"),
             F.first("Month").alias("MonthNum"),
             F.first("MonthName").alias("MonthName"),
         )
         .withColumn("Value", F.lit(1))  # for compatibility with auto-date tables
)

dim_month_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_MONTH_TABLE)
print(f"✓ Created {DIM_MONTH_TABLE} with {dim_month_df.count():,} rows")

## Step 4 · Validate coverage

Ensure every YearMonth in `focus_partitioned` exists in `dim_month`.

In [ ]:
# Get distinct months from focus_partitioned
focus_months = (
    spark.table(SOURCE_TABLE)
         .select(F.date_format("ChargePeriodStart", "yyyy-MM").alias("YearMonth"))
         .distinct()
)

dim_month = spark.table(DIM_MONTH_TABLE)

# Anti-join to find gaps
gaps = focus_months.join(dim_month, on="YearMonth", how="left_anti")
gap_count = gaps.count()

if gap_count > 0:
    print(f"⚠️  Coverage gap detected: {gap_count} YearMonth(s) in focus_partitioned missing from dim_month")
    gaps.show()
    raise ValueError("dim_month coverage incomplete!")
else:
    focus_month_count = focus_months.count()
    print(f"✓ Coverage validated: all {focus_month_count} YearMonth in focus_partitioned exist in dim_month")
    print(f"  {DIM_DATE_TABLE} rows: {spark.table(DIM_DATE_TABLE).count():,}")
    print(f"  {DIM_MONTH_TABLE} rows: {dim_month.count():,}")